In [3]:
from pynq import Overlay
from pynq import MMIO

# Load your bitstream overlay
ol = Overlay('design.bit')

# BRAM write (keep as is)
BRAM_BASE = 0x40000000          # use your actual BRAM base address
bram_mmio = MMIO(BRAM_BASE, 0x10000)
test_data = [0x0, 0x0, 
0x00400313,
0x01000393,
0x00000493,
0x00736433,
0x00147413,
0x00041a63,
0x00135313,
0x0013d393,
0x00148493,
0xfe9ff06f,
0x00137413, 
0x00040463,
0x00c0006f,
0x00135313,
0xff1ff06f,
0x0013f413, 
0x02040063,
0x0063f863,
0x00030513, 
0x00038313,
0x00050393,
0x406383b3,
0x00038863,
0xe1ff06f,
0x0013d393, 
0x00931333,
0x00603023]

for i, val in enumerate(test_data):
    bram_mmio.write(i * 4, val) 

# AXI GPIO (keep as is)
gpio = AxiGPIO(ol.ip_dict['axi_gpio_0'])
channel = gpio.channel1

results = [] 
for pc in range(len(test_data)): 
    time.sleep(1) 
    val = channel.read() 
    results.append(val) 
    print(f'Reading result at PC={pc}: 0x{val:08X}')




def read_concat_regs(reg0, reg1, num_regs=32, reg_size=4):
    reg0_mmio = reg0.mmio
    reg1_mmio = reg1.mmio
    concatenated_regs = []
    for i in range(num_regs):
        val0 = reg0_mmio.read(i * reg_size)
        val1 = reg1_mmio.read(i * reg_size)
        concatenated = (val1 << 32) | val0
        concatenated_regs.append(concatenated)
    return concatenated_regs

# Base addresses of BRAMs 1 to 8
bram_bases = [
    ol.axi_bram_ctrl_1.mmio.base_addr,
    ol.axi_bram_ctrl_2.mmio.base_addr,
    ol.axi_bram_ctrl_3.mmio.base_addr,
    ol.axi_bram_ctrl_4.mmio.base_addr,
    ol.axi_bram_ctrl_5.mmio.base_addr,
    ol.axi_bram_ctrl_6.mmio.base_addr,
    ol.axi_bram_ctrl_7.mmio.base_addr,
    ol.axi_bram_ctrl_8.mmio.base_addr,
]

bram_size_words = 256
word_size = 4

bram_mmios = [MMIO(base, bram_size_words * word_size) for base in bram_bases]

data_memory = [[] for _ in range(8)]
for bram_index, bram_mmio in enumerate(bram_mmios):
    for i in range(bram_size_words):
        val = bram_mmio.read(i * word_size)
        data_memory[bram_index].append(val & 0xFF)

reg_ip0 = ol.reg_ip_0
reg_ip1 = ol.reg_ip_1

concat_regs = read_concat_regs(reg_ip0, reg_ip1)

# Print concatenated registers in columns
NUM_COLS = 4
print("Registers contents")
header = ""
for col in range(NUM_COLS):
    header += f"{'Addr':>5} {'Value':>18}   "
print(header)

for i in range(0, len(concat_regs), NUM_COLS):
    line = ""
    for col in range(NUM_COLS):
        if i + col < len(concat_regs):
            addr = i + col
            val = concat_regs[addr]
            line += f"{addr:5} 0x{val:016X}   "
    print(line)

print("\nData Memory Contents")
# Print BRAM header
print("Addr ", end='')

print("\n")
# Print data memory rows for each address
for i in range(bram_size_words):
    print(f"{i:04}: ", end='')
    for bram_idx in range(7, -1, -1):
        print(f"{data_memory[bram_idx][i]:02X}   ", end='')
    print()


Reading result at PC=0: 0x00000004
Reading result at PC=1: 0x00000010
Reading result at PC=2: 0x00000000
Reading result at PC=3: 0x00000014
Reading result at PC=4: 0x00000000
Reading result at PC=5: 0x00000000
Reading result at PC=6: 0x00000002
Reading result at PC=7: 0x00000008
Reading result at PC=8: 0x00000001
Reading result at PC=9: 0x00000030
Reading result at PC=10: 0x0000000A
Reading result at PC=11: 0x00000000
Reading result at PC=12: 0x00000000
Reading result at PC=13: 0x00000001
Reading result at PC=14: 0x00000004
Reading result at PC=15: 0x00000002
Reading result at PC=16: 0x00000030
Reading result at PC=17: 0x00000005
Reading result at PC=18: 0x00000001
Reading result at PC=19: 0x00000000
Reading result at PC=20: 0x00000001
Reading result at PC=21: 0x00000000
Reading result at PC=22: 0x0000003C
Reading result at PC=23: 0x00000000
Reading result at PC=24: 0x00000000
Reading result at PC=25: 0x00000002
Reading result at PC=26: 0x00000004
Reading result at PC=27: 0x00000000
Re